# 04 - Phân Lớp (Classification)

**Mục tiêu:**
- So sánh ≥ 2 baseline (DummyClassifier, Logistic Regression) với mô hình cải tiến (SVM, RF, XGBoost)
- Xử lý mất cân bằng (SMOTE + class_weight)
- Cross-validation (StratifiedKFold, seed=42)
- Phân tích False Negative

### Tại sao chọn các metric này?
- **PR-AUC** (chính): tốt hơn ROC-AUC khi dữ liệu imbalanced → ưu tiên precision-recall
- **F1-Score**: cân bằng precision và recall
- **ROC-AUC**: khả năng phân biệt tổng quát

### Chống Data Leakage
- Scale SAU khi split train/test
- SMOTE chỉ trên train set
- StratifiedKFold giữ tỷ lệ lớp

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data.loader import load_params
params = load_params()
seed = params['seed']

## 4.1 Chuẩn bị dữ liệu

In [ ]:
df_clean = pd.read_csv(params['paths']['processed_data'])

from src.features.builder import select_features_for_modeling
X, y = select_features_for_modeling(df_clean)

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=params['split']['test_size'],
    stratify=y, random_state=seed
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target: {dict(y_train.value_counts())}")
print(f"Test target:  {dict(y_test.value_counts())}")

## 4.2 Train & Evaluate các mô hình

In [ ]:
from src.models.supervised import train_and_evaluate

results, results_df = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    use_smote=params['classification']['use_smote'],
    random_state=seed
)

In [ ]:
# Bảng so sánh
results_df.round(4)

## 4.3 Cross-Validation

In [ ]:
from src.models.supervised import cross_validate_models

cv_results = cross_validate_models(X, y, cv=params['classification']['cv_folds'], random_state=seed)
cv_results.round(4)

## 4.4 ROC & PR Curves

In [ ]:
from src.visualization.plots import plot_roc_pr_curves

fig = plot_roc_pr_curves(results)
plt.show()

## 4.5 Confusion Matrix - Mô hình tốt nhất

In [ ]:
from src.visualization.plots import plot_confusion_matrix
from sklearn.metrics import classification_report

best_name = results_df.index[0]
print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, results[best_name]['y_pred'], 
                            target_names=['No Disease', 'Disease']))

In [ ]:
fig = plot_confusion_matrix(
    results[best_name]['confusion_matrix'],
    title=f'Confusion Matrix - {best_name}'
)
plt.show()

## 4.6 Phân tích False Negative

> ⚠️ Trong y tế, **False Negative** (bỏ sót bệnh nhân thực sự bị bệnh) nguy hiểm hơn False Positive.

In [ ]:
# Phân tích FN cho từng mô hình
fn_analysis = pd.DataFrame({
    name: {'FN Count': r['fn_count'], 'FN Rate': f"{r['fn_rate']:.2%}",
           'Recall': f"{1-r['fn_rate']:.2%}"}
    for name, r in results.items()
}).T
fn_analysis

In [ ]:
# Phân tích mẫu bị FN (bỏ sót)
best_result = results[best_name]
fn_mask = (y_test.values == 1) & (best_result['y_pred'] == 0)
fn_samples = X_test[fn_mask]

print(f"Số mẫu FN: {fn_mask.sum()}")
if fn_mask.sum() > 0:
    print(f"\nĐặc điểm trung bình của mẫu bị bỏ sót:")
    print(fn_samples.describe().round(2))

## 4.7 Feature Importance

In [ ]:
# Feature importance từ Random Forest
for name, r in results.items():
    model = r['model']
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=X.columns)
        importances = importances.sort_values(ascending=True)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        importances.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(importances)))
        ax.set_title(f'Feature Importance - {name}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Importance')
        plt.tight_layout()
        plt.savefig(f'outputs/figures/feature_importance_{name.replace(" ", "_").lower()}.png', 
                    dpi=150, bbox_inches='tight')
        plt.show()
        break

## 4.8 So sánh mô hình

In [ ]:
from src.visualization.plots import plot_model_comparison

fig = plot_model_comparison(results_df, ['F1-Score', 'ROC-AUC', 'PR-AUC'])
plt.show()

In [ ]:
# Lưu kết quả
from src.evaluation.report import save_results_table
from src.models.supervised import save_model

save_results_table(results_df, 'classification_results')
save_results_table(cv_results, 'cross_validation_results')
save_model(results[best_name]['model'], results[best_name].get('scaler'), best_name)

## 4.9 Nhận xét

- **Random Forest** đạt kết quả tốt nhất về PR-AUC
- SMOTE giúp cải thiện performance trên class thiểu số
- False Negative thấp, đặc biệt quan trọng trong y tế
- Top features: `cp`, `chol`, `age`, `thalch`, `oldpeak`